# Healthcare Financial Responsibility Analysis

## Project Overview

I work with patient registration and insurance information in my current job, so I wanted to build a project around a question that is close to what I see at work.

For this project, I use synthetic healthcare data and look at patient financial responsibility. I mainly want to know if some payer and visit type combinations are more likely to leave the patient with a higher amount to pay.

The project includes:

1. Business Problem
2. Source Data Check
3. OMOP ETL Pipeline
4. Business Analysis
5. Business Insights and Recommendations

## 1. Business Problem

### Business Question

Which payer and visit type combinations are associated with higher estimated patient financial responsibility?

### Why I chose this question

In my daily work, I deal with insurance and registration information before patient visits. Epic can check insurance eligibility, and we also work with self-pay patients and cost estimates.

But a patient can have active insurance and still need to pay a lot. I wanted to look at this part more closely.

My idea is to use the historical payer and visit information to see if there are some patterns in patient responsibility.

### Approach

I use total cost and payer coverage to estimate how much is left for the patient.

Before doing the analysis, I first check the source data, especially the financial fields and the relationships between encounters, patients and payers. Then I use the OHDSI ETL pipeline to convert the data into OMOP format.

## 2. Source Data Understanding & Validation

Before I started the ETL, I wanted to understand what is actually in the source data and where the fields I need are stored.

I mainly looked at three files:

- **Patients** - patient information and patient ID
- **Encounters** - visit information, payer, claim cost and payer coverage
- **Payers** - payer ID and payer name

`encounters` is the main file for this project because most of the fields I need are already there. The other files help connect the encounter to the patient and payer.

### 2.1 Environment and Data Source

The data comes from the OHDSI Tutorial-ETL project. The source data was generated by Synthea, so it is synthetic patient data, not real patient records.



In [5]:
!pip install duckdb

In [6]:
import duckdb

print(duckdb.__version__)

1.3.2


In [7]:
!git clone https://github.com/OHDSI/Tutorial-ETL.git

Cloning into 'Tutorial-ETL'...
remote: Enumerating objects: 2007, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 2007 (delta 12), reused 54 (delta 10), pack-reused 1948 (from 1)
Receiving objects: 100% (2007/2007), 134.74 MiB | 13.54 MiB/s, done.
Resolving deltas: 100% (1281/1281), done.
Updating files: 100% (1359/1359), done.


In [8]:
import os

data_path = "/content/Tutorial-ETL/data/syntheaRaw"

os.listdir(data_path)

['claims.csv',
 'allergies.csv',
 'careplans.csv',
 'immunizations.csv',
 'imaging_studies.csv',
 'payer_transitions.csv',
 'devices.csv',
 'procedures.csv',
 'patients.csv',
 'supplies.csv',
 'conditions.csv',
 'claims_transactions.csv',
 'payers.csv',
 'encounters.csv',
 'providers.csv',
 'observations.csv',
 'medications.csv',
 'organizations.csv']

### 2.2 Source Schema and Key Fields

I started by checking the columns in the three source files.

The main relationships I found are:

- `encounters.PATIENT` → `patients.Id`
- `encounters.PAYER` → `payers.Id`

For my analysis, the most important fields are `TOTAL_CLAIM_COST`, `PAYER_COVERAGE` and `ENCOUNTERCLASS`.

At this point I mainly wanted to make sure I understand the IDs and financial fields before I start joining or transforming anything.

In [9]:
con = duckdb.connect()

con.sql("""
select *
from read_csv_auto('/content/Tutorial-ETL/data/syntheaRaw/patients.csv')
limit 5
""").df()

,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,MIDDLE,LAST,...,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME
0,07cb2889-a47c-bede-f317-57a3822ec014,1992-05-18,NaT,999-69-2367,S99936489,X66405296X,Mr.,Herbert830,None,Franecki195,...,Avenel,New Jersey,Middlesex County,34023,07001,40.596859,-74.232102,48949.28,21541.34,28036
1,62cffc53-b831-0764-6142-746174fd42f9,1993-10-16,NaT,999-22-9345,S99966248,X48462820X,Mr.,Daron260,Lanny564,Volkman526,...,Barrington,New Jersey,Camden County,34007,08033,39.858504,-75.074387,127167.31,6173.76,59158
2,96d3bead-085b-7b6d-731d-a87f87f122b5,2006-03-08,NaT,999-44-9518,S99992362,None,Ms.,Katherina205,None,Hettinger594,...,Tabernacle,New Jersey,Burlington County,<NA>,00000,39.791010,-74.627516,84588.59,145112.75,112005
3,7fa095d4-903c-0ef7-6e31-22bc2abd4b0d,1997-07-02,NaT,999-99-6717,S99925417,X86369024X,Mr.,George991,Ken316,Schneider199,...,Pompton Lakes,New Jersey,Passaic County,34031,07442,41.039472,-74.293212,90700.02,943833.73,929749
4,d110dbfd-52f7-95b0-f686-03c61e99ec3c,1995-06-28,NaT,999-73-9216,S99958762,X27610089X,Mrs.,Ewa95,None,Hoeger474,...,Jersey City,New Jersey,Hudson County,34017,07302,40.735510,-74.062300,33742.35,350484.56,23420


In [10]:
# what fields do I have here?
con.sql("""
DESCRIBE select *
from read_csv_auto('/content/Tutorial-ETL/data/syntheaRaw/patients.csv')
""").df()

,column_name,column_type,null,key,default,extra
0,Id,VARCHAR,YES,None,None,None
1,BIRTHDATE,DATE,YES,None,None,None
2,DEATHDATE,DATE,YES,None,None,None
3,SSN,VARCHAR,YES,None,None,None
4,DRIVERS,VARCHAR,YES,None,None,None
5,PASSPORT,VARCHAR,YES,None,None,None
6,PREFIX,VARCHAR,YES,None,None,None
7,FIRST,VARCHAR,YES,None,None,None
8,MIDDLE,VARCHAR,YES,None,None,None
9,LAST,VARCHAR,YES,None,None,None


In [11]:
con.sql("""
select *
from read_csv_auto('/content/Tutorial-ETL/data/syntheaRaw/payers.csv')
limit 5
""").df()

,Id,NAME,OWNERSHIP,ADDRESS,CITY,STATE_HEADQUARTERED,ZIP,PHONE,AMOUNT_COVERED,AMOUNT_UNCOVERED,...,UNCOVERED_ENCOUNTERS,COVERED_MEDICATIONS,UNCOVERED_MEDICATIONS,COVERED_PROCEDURES,UNCOVERED_PROCEDURES,COVERED_IMMUNIZATIONS,UNCOVERED_IMMUNIZATIONS,UNIQUE_CUSTOMERS,QOLS_AVG,MEMBER_MONTHS
0,a735bf55-83e9-331a-899d-a82a60b9f60c,Medicare,GOVERNMENT,None,None,None,None,None,5081328.41,294335.08,...,0,1596,0,2895,0,411,0,28,0.710513,3960
1,df166300-5a78-3502-a46a-832842197811,Medicaid,GOVERNMENT,None,None,None,None,None,9633327.62,153540.18,...,0,440,0,3250,0,824,0,25,0.937905,6228
2,d18ef2e6-ef40-324c-be54-34a5ee865625,Dual Eligible,GOVERNMENT,None,None,None,None,None,418063.43,3034.79,...,0,78,0,110,0,26,0,3,0.704604,240
3,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,Humana,PRIVATE,None,None,None,None,None,11162140.32,3643136.64,...,0,1553,0,4669,0,1220,0,30,0.942510,9936
4,b046940f-1664-3047-bca7-dfa76be352a4,Blue Cross Blue Shield,PRIVATE,None,None,None,None,None,2559623.17,805220.14,...,0,435,0,1173,0,394,0,39,0.385267,5628


In [12]:
# Check the columns and data types in the payers dataset.

con.sql("""
describe
select * from read_csv_auto('/content/Tutorial-ETL/data/syntheaRaw/payers.csv')
""").df()

,column_name,column_type,null,key,default,extra
0,Id,VARCHAR,YES,None,None,None
1,NAME,VARCHAR,YES,None,None,None
2,OWNERSHIP,VARCHAR,YES,None,None,None
3,ADDRESS,VARCHAR,YES,None,None,None
4,CITY,VARCHAR,YES,None,None,None
5,STATE_HEADQUARTERED,VARCHAR,YES,None,None,None
6,ZIP,VARCHAR,YES,None,None,None
7,PHONE,VARCHAR,YES,None,None,None
8,AMOUNT_COVERED,DOUBLE,YES,None,None,None
9,AMOUNT_UNCOVERED,DOUBLE,YES,None,None,None


In [13]:
# encounters is the main one, so I checked this more carefully
con.sql("""
DESCRIBE
select *
from read_csv_auto('/content/Tutorial-ETL/data/syntheaRaw/encounters.csv')
""").df()

,column_name,column_type,null,key,default,extra
0,Id,VARCHAR,YES,None,None,None
1,START,TIMESTAMP WITH TIME ZONE,YES,None,None,None
2,STOP,TIMESTAMP WITH TIME ZONE,YES,None,None,None
3,PATIENT,VARCHAR,YES,None,None,None
4,ORGANIZATION,VARCHAR,YES,None,None,None
5,PROVIDER,VARCHAR,YES,None,None,None
6,PAYER,VARCHAR,YES,None,None,None
7,ENCOUNTERCLASS,VARCHAR,YES,None,None,None
8,CODE,BIGINT,YES,None,None,None
9,DESCRIPTION,VARCHAR,YES,None,None,None


In [14]:
con.sql("""
select ID, PATIENT, START, ENCOUNTERCLASS, PAYER,
       BASE_ENCOUNTER_COST, TOTAL_CLAIM_COST, PAYER_COVERAGE
from read_csv_auto('/content/Tutorial-ETL/data/syntheaRaw/encounters.csv')
limit 10
""").df()

,Id,PATIENT,START,ENCOUNTERCLASS,PAYER,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE
0,57d6cc36-4f34-b14f-405d-83635d410871,07cb2889-a47c-bede-f317-57a3822ec014,2001-05-28 14:58:37+00:00,ambulatory,0133f751-9229-3cfd-815f-b6d4979bdd6a,84.76,207.23,0.00
1,d6618e6d-fc9b-de67-9c5f-47e3738557a2,62cffc53-b831-0764-6142-746174fd42f9,1997-10-18 11:40:07+00:00,inpatient,e03e23c9-4df1-3eb6-a62d-f70f02301496,143.38,987.83,0.00
2,e28c5d5d-8228-02b4-7141-219b74ab8a92,96d3bead-085b-7b6d-731d-a87f87f122b5,2006-03-08 09:09:12+00:00,wellness,b046940f-1664-3047-bca7-dfa76be352a4,179.08,446.10,0.00
3,83d48470-7e7c-41a9-21c3-2de452ae6480,7fa095d4-903c-0ef7-6e31-22bc2abd4b0d,1998-01-17 21:46:39+00:00,ambulatory,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,84.76,84.76,0.00
4,28b8e33c-28f5-720f-a7cc-cde111ce2fa9,d110dbfd-52f7-95b0-f686-03c61e99ec3c,2005-07-06 23:26:03+00:00,wellness,df166300-5a78-3502-a46a-832842197811,179.08,921.18,771.18
5,0789774a-ef70-89fe-9546-18608d24a551,07cb2889-a47c-bede-f317-57a3822ec014,2005-06-13 14:58:37+00:00,wellness,0133f751-9229-3cfd-815f-b6d4979bdd6a,179.08,1193.18,0.00
6,974f7b8c-3bdf-850f-eccb-bca34ece5fe6,d110dbfd-52f7-95b0-f686-03c61e99ec3c,2006-07-12 23:26:03+00:00,wellness,df166300-5a78-3502-a46a-832842197811,179.08,1883.79,1783.79
7,a11c0a94-5c77-c3db-2d4e-77ab4d6a29a6,07cb2889-a47c-bede-f317-57a3822ec014,2006-06-10 21:58:37+00:00,ambulatory,0133f751-9229-3cfd-815f-b6d4979bdd6a,84.76,84.76,0.00
8,5d26eecd-27fa-2838-a338-04dcf4a722c4,07cb2889-a47c-bede-f317-57a3822ec014,2006-06-19 14:58:37+00:00,wellness,0133f751-9229-3cfd-815f-b6d4979bdd6a,179.08,2107.04,0.00
9,31916f45-432f-9938-6174-cbc52bc42c97,62cffc53-b831-0764-6142-746174fd42f9,2001-01-06 11:06:33+00:00,ambulatory,e03e23c9-4df1-3eb6-a62d-f70f02301496,84.76,84.76,0.00


### 2.3 Basic Source Profile

After looking at the columns, I wanted to get a basic idea of the data size and visit mix.

I also checked the cost range. This matters because if the cost values are very spread out, average alone may not tell the full story later.

In [15]:
con.sql("""
select ENCOUNTERCLASS, count(*) encounter_count
from read_csv_auto('/content/Tutorial-ETL/data/syntheaRaw/encounters.csv')
GROUP BY ENCOUNTERCLASS
order by encounter_count desc
""").df()

,ENCOUNTERCLASS,encounter_count
0,ambulatory,6284
1,wellness,1973
2,outpatient,1567
3,urgentcare,369
4,emergency,348
5,inpatient,111
6,snf,36
7,virtual,12
8,hospice,10
9,home,5


In [16]:
con.sql("""
SELECT count(*) total_encounters,
 min(TOTAL_CLAIM_COST) min_claim_cost,
 max(TOTAL_CLAIM_COST) max_claim_cost,
 avg(TOTAL_CLAIM_COST) avg_claim_cost,
 min(PAYER_COVERAGE) min_payer_coverage,
 max(PAYER_COVERAGE) max_payer_coverage,
 avg(PAYER_COVERAGE) avg_payer_coverage
FROM read_csv_auto('/content/Tutorial-ETL/data/syntheaRaw/encounters.csv')
""").df()

,total_encounters,min_claim_cost,max_claim_cost,avg_claim_cost,min_payer_coverage,max_payer_coverage,avg_payer_coverage
0,10715,75.0,95814.54,3434.482743,0.0,95764.54,2470.643182


There are 10,715 encounters. Most of them are ambulatory, wellness and outpatient visits.

The cost range is pretty wide. Claim cost starts at $75 and goes up to around 95,815. Because of this, later I want to look at median together with average. A few expensive visits could make the average look much higher.

### 2.4 Financial Data Validation

Patient responsibility will be based on total cost minus payer coverage, so I checked these two fields before using them.

I mainly wanted to know:

- Are there missing values?
- Are there encounters with zero cost?
- Does payer coverage ever become larger than the total cost?
- How often is payer coverage zero?

In [17]:
con.sql("""
select count(*) total_encounters,
 sum(case when PAYER_COVERAGE > TOTAL_CLAIM_COST then 1 else 0 end) coverage_exceeds_claim,
 sum(case when PAYER_COVERAGE = 0 then 1 else 0 end) zero_coverage,
 sum(case when TOTAL_CLAIM_COST = 0 then 1 else 0 end) zero_cost,
 sum(case when TOTAL_CLAIM_COST is null or PAYER_COVERAGE is null then 1 else 0 end) missing_values
from read_csv_auto('/content/Tutorial-ETL/data/syntheaRaw/encounters.csv')
""").df()

,total_encounters,coverage_exceeds_claim,zero_coverage_encounters,zero_cost_encounters,missing_financial_values
0,10715,0.0,3190.0,0.0,0.0


#### What I found

The financial fields look fine for this analysis. I did not find missing financial values, zero claim costs, or payer coverage higher than the claim cost.

3,190 encounters have zero payer coverage, about 29.8% of all encounters.

### 2.5 Relationship Validation

The next thing I checked was the IDs.

If an encounter has a patient ID or payer ID that cannot be found in the other tables, I could lose records when I join the data later. So I checked the matches first.

In [18]:
con.sql("""
select count(*) total_encounters,
 sum(case when p.Id is null then 1 else 0 end) unmatched_patient_ids
from read_csv_auto('/content/Tutorial-ETL/data/syntheaRaw/encounters.csv') e
left join read_csv_auto('/content/Tutorial-ETL/data/syntheaRaw/patients.csv') p
 on e.PATIENT = p.Id
""").df()

,total_encounters,unmatched_patient_ids
0,10715,0.0


All 10,715 encounters matched to a patient record.

In [19]:
# Check whether encounter payer IDs can be matched to the payers table
payer_match = con.sql("""
SELECT count(*) total_encounters,
       sum(case when p.Id is null then 1 else 0 end) unmatched_payer_ids
FROM read_csv_auto('/content/Tutorial-ETL/data/syntheaRaw/encounters.csv') e
LEFT JOIN read_csv_auto('/content/Tutorial-ETL/data/syntheaRaw/payers.csv') p
     ON e.PAYER = p.Id
""").df()

payer_match

,total_encounters,unmatched_payer_ids
0,10715,0.0


All encounter records matched to both the patients and payers tables. So these tables can be joined for the ETL process.

## 3. OMOP ETL Pipeline

After checking the source data, I used the OHDSI ETL pipeline to convert the Synthea data into the OMOP Common Data Model (CDM).

I then checked the OMOP tables and used the standardized data for the financial analysis.

### 3.1 Convert Synthea Data to OMOP CDM

Run the OHDSI Synthea-to-OMOP ETL pipeline to convert the raw healthcare data into OMOP CDM tables. I used the SQLMesh version because it works with SQL and DuckDB, which also fits better with this project.


In [20]:
etl_path = "/content/Tutorial-ETL/etl"
os.listdir(etl_path)

['sqlmesh-synthea', 'etl-synthea', 'dbt-synthea']


In [42]:
# Install uv, the Python environment/package manager used by this SQLMesh project.
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 47.7 MB/s eta 0:00:00


In [43]:
# Go to the SQLMesh tutorial project folder.
%cd /content/Tutorial-ETL/etl/sqlmesh-synthea

# Install the dependencies defined by this project.
!uv pip install -e .

/content/Tutorial-ETL/etl/sqlmesh-synthea
Using Python 3.13.15 environment at: /usr
Resolved 123 packages in 1.45s
  × Failed to build `sqlmesh-synthea-tutorial @
  │ file:///content/Tutorial-ETL/etl/sqlmesh-synthea`
  ├─▶ The build backend returned an error
  ╰─▶ Call to `setuptools.build_meta:__legacy__.build_editable` failed (exit
      status: 1)

      [stderr]
      error: Multiple top-level packages discovered in a flat-layout:
      ['seeds', 'models', 'audits'].

      To avoid accidental inclusion of unwanted files or directories,
      setuptools will not proceed with this build.

      If you are trying to create a single distribution with multiple packages
      on purpose, you should not rely on automatic discovery.
      Instead, consider the following options:

      1. set up custom discovery (`find` directive with `include` or
      `exclude`)
      2. use a `src-layout`
      3. explicitly set `py_modules` or `packages` with a list of names

      To find more inform

In [44]:
# Show the dependency configuration for the SQLMesh project.
!cat pyproject.toml

[project]
name = "sqlmesh-synthea-tutorial"
version = "0.1.0"
description = "Tutorial ETL pipeline that loads Synthea into OMOP v5.4 using SQLMesh"
readme = "README.md"
requires-python = ">=3.10"
authors = [
    { name = "OHDSI" }
]
dependencies = [
    "sqlmesh[duckdb,pandas,web]>=0.200.0,<1.0",
    "duckdb>=1.4.0"
]


In [45]:
# Install the dependencies needed by the SQLMesh OMOP pipeline.
!uv pip install "sqlmesh[duckdb,pandas,web]>=0.200.0,<1.0" "duckdb>=1.4.0"

Using Python 3.13.15 environment at: /usr
Resolved 122 packages in 80ms
Uninstalled 5 packages in 32ms
Installed 15 packages in 248ms
 + croniter==6.2.4
 + dateparser==1.2.1
 - duckdb==1.3.2
 + duckdb==1.5.5
 - fastapi==0.141.1
 + fastapi==0.120.1
 + hyperscript==0.3.0
 + jedi==0.20.0
 + json-stream==2.5.1
 + json-stream-rs-tokenizer==0.5.3
 + ruamel-yaml==0.19.1
 - sqlglot==25.20.2
 + sqlglot==30.8.0
 + sqlmesh==0.236.1
 + sse-starlette==3.4.11
 - starlette==1.6.0
 + starlette==0.49.3
 + time-machine==3.5.0
 - uvicorn==0.52.4
 + uvicorn==0.22.0


In [46]:
# Check whether SQLMesh can read this project and connect to DuckDB.
!sqlmesh info

Models: 73
Macros: 0
Data warehouse connection succeeded


In [47]:
# Build the OMOP tables in the SQLMesh development environment.
!sqlmesh plan dev

Linter errors for 
/content/Tutorial-ETL/etl/sqlmesh-synthea/models/omop/device_exposure.sql:
 - nomissingunittest: Model omop.device_exposure is missing unit test(s). Please
add in the tests/ directory.
Linter errors for 
/content/Tutorial-ETL/etl/sqlmesh-synthea/models/omop/cdm_source.sql:
 - nomissingunittest: Model omop.cdm_source is missing unit test(s). Please add 
in the tests/ directory.
Linter errors for 
/content/Tutorial-ETL/etl/sqlmesh-synthea/models/omop/note_nlp.sql:
 - nomissingunittest: Model omop.note_nlp is missing unit test(s). Please add in
the tests/ directory.
Linter errors for 
/content/Tutorial-ETL/etl/sqlmesh-synthea/models/omop/drug_era.sql:
 - nomissingunittest: Model omop.drug_era is missing unit test(s). Please add in
the tests/ directory.
Linter errors for 
/content/Tutorial-ETL/etl/sqlmesh-synthea/models/omop/condition_era.sql:
 - nomissingunittest: Model omop.condition_era is missing unit test(s). Please 
add in the tests/ directory.
Linter errors for 
/

SQLMesh could read the project, but the configuration stopped the plan because of a missing unit-test lint rule. I ignored that lint rule and ran the plan again.

In [48]:
# Show the SQLMesh project configuration.
!cat config.yaml

gateways:
  localduck:
    connection:
      type: duckdb
      database: sqlmesh-synthea.duckdb

default_gateway: localduck

model_defaults:
  dialect: duckdb

format:
  max_text_width: 140
  normalize_functions: upper
  leading_comma: false
  indent: 4
  pad: 2

linter:
  enabled: true
  rules: "ALL"
  ignored_rules: ["noselectstar", "nomissingaudits"]


In [50]:
# Ignore the missing unit test lint rule for this tutorial project.

from pathlib import Path

config_path = Path("config.yaml")
config = config_path.read_text()

config = config.replace(
    'ignored_rules: ["noselectstar", "nomissingaudits"]',
    'ignored_rules: ["noselectstar", "nomissingaudits", "nomissingunittest"]'
)

config_path.write_text(config)

print(config_path.read_text())

gateways:
  localduck:
    connection:
      type: duckdb
      database: sqlmesh-synthea.duckdb

default_gateway: localduck

model_defaults:
  dialect: duckdb

format:
  max_text_width: 140
  normalize_functions: upper
  leading_comma: false
  indent: 4
  pad: 2

linter:
  enabled: true
  rules: "ALL"
  ignored_rules: ["noselectstar", "nomissingaudits", "nomissingunittest"]



In [53]:
# Apply the SQLMesh plan
!sqlmesh plan dev --auto-apply

流式输出内容被截断，只能显示最后 5000 行内容。
 '0' '0' '0' '1' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '1' '0' '0' '0'
 '0' '0' '0' '1' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0'
 '0' '0' '0' '0' '0' '0' '0' '0' '0' '1' '0' '0' '0' '1' '1' '0' '1' '0'
 '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0'
 '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0'
 '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '1' '0' '0' '1' '0' '1'
 '0' '0' '0' '0' '0' '0' '1' '0' '0' '0' '1' '0' '0' '0' '0' '0' '0' '0'
 '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '1' '0' '0' '0' '1'
 '0' '0' '0' '1' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0'
 '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0'
 '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '1' '0' '0'
 '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '0' '1' '1' '0'
 '0' '0' '0' '0' '0' '0' '1' '0' '0' '0' '1' '0' '0' '1' '0' '0' '0' '0'
 '0' '0' '0' '0' '0' '0'

### 3.2 Check the OMOP Tables

After the ETL finished, I checked where the fields for my analysis ended up.

For this project I use:

- `VISIT_OCCURRENCE` for standardized visit information
- `COST` for the financial fields
- `stg.encounters` to connect the OMOP visit back to its payer
- `stg.payers` to get the payer name

`PERSON` was also created by the pipeline, but I do not need patient demographics for the question I am answering here.


In [56]:
# Preview the standardized OMOP VISIT_OCCURRENCE table.

omop_con.sql(""" SELECT *
  FROM omop__dev.visit_occurrence
  LIMIT 5
""").df()

,visit_occurrence_id,person_id,visit_concept_id,visit_start_date,visit_start_datetime,visit_end_date,visit_end_datetime,visit_type_concept_id,provider_id,care_site_id,visit_source_value,visit_source_concept_id,admitted_from_concept_id,admitted_from_source_value,discharged_to_concept_id,discharged_to_source_value,preceding_visit_occurrence_id
0,1530,12,9202,2013-10-12,2013-10-12 14:53:22,2013-10-12,2013-10-12 15:08:22,32827,233,<NA>,b66bf1ae-2fde-14d9-a90e-07a8cb36ff01,0,<NA>,None,<NA>,None,<NA>
1,1531,12,9202,2013-11-16,2013-11-16 14:53:22,2013-11-16,2013-11-16 15:08:22,32827,233,<NA>,1f3c9d4d-809a-c17e-40b6-2dd1503e3ca5,0,<NA>,None,<NA>,None,1530
2,1532,12,9202,2014-01-18,2014-01-18 14:53:22,2014-01-18,2014-01-18 15:08:22,32827,233,<NA>,8520326c-af6c-0d59-dd95-81c0229a686a,0,<NA>,None,<NA>,None,1531
3,1533,12,9202,2014-03-22,2014-03-22 14:53:22,2014-03-22,2014-03-22 15:08:22,32827,233,<NA>,2c031685-932f-31a4-15ea-936e2461f441,0,<NA>,None,<NA>,None,1532
4,1534,12,9203,2014-04-21,2014-04-21 14:53:22,2014-04-21,2014-04-21 18:20:22,32827,242,<NA>,b7c5b33e-cae7-ed59-d1a0-74f31b358339,0,<NA>,None,<NA>,None,1533


In [57]:
# Show the OMOP VISIT_OCCURRENCE table.

omop_con.sql("""
DESCRIBE omop__dev.visit_occurrence
""").df()

,column_name,column_type,null,key,default,extra
0,visit_occurrence_id,BIGINT,YES,None,None,None
1,person_id,BIGINT,YES,None,None,None
2,visit_concept_id,INTEGER,YES,None,None,None
3,visit_start_date,DATE,YES,None,None,None
4,visit_start_datetime,TIMESTAMP,YES,None,None,None
5,visit_end_date,DATE,YES,None,None,None
6,visit_end_datetime,TIMESTAMP,YES,None,None,None
7,visit_type_concept_id,INTEGER,YES,None,None,None
8,provider_id,BIGINT,YES,None,None,None
9,care_site_id,INTEGER,YES,None,None,None


In [60]:
omop_con.sql("""
DESCRIBE omop__dev.cost
""").df()

,column_name,column_type,null,key,default,extra
0,cost_id,BIGINT,YES,None,None,None
1,cost_event_id,BIGINT,YES,None,None,None
2,cost_domain_id,VARCHAR,YES,None,None,None
3,cost_type_concept_id,INTEGER,YES,None,None,None
4,currency_concept_id,INTEGER,YES,None,None,None
5,total_charge,DOUBLE,YES,None,None,None
6,total_cost,DOUBLE,YES,None,None,None
7,total_paid,DOUBLE,YES,None,None,None
8,paid_copay,DOUBLE,YES,None,None,None
9,paid_coinsurance,DOUBLE,YES,None,None,None


In [63]:
# Check which OMOP domains are stored in the COST table.

omop_con.sql("""
    SELECT cost_domain_id,COUNT(*) AS record_count
    FROM omop__dev.cost
    GROUP BY cost_domain_id
    ORDER BY record_count DESC
""").df()

,cost_domain_id,record_count
0,Visit Occurrence,10652


In [68]:
omop_con.sql("""
DESCRIBE stg__dev.payers
""").df()

,column_name,column_type,null,key,default,extra
0,payer_id,VARCHAR,YES,None,None,None
1,name,VARCHAR,YES,None,None,None
2,ownership,VARCHAR,YES,None,None,None
3,city,VARCHAR,YES,None,None,None
4,state,VARCHAR,YES,None,None,None
5,zip,VARCHAR,YES,None,None,None
6,phone,VARCHAR,YES,None,None,None


#### Conclusion

For this analysis, I use two OMOP tables and two staging tables.

**`VISIT_OCCURRENCE`**

This table provides the standardized visit information. I use:

- `visit_occurrence_id` to identify each visit and connect it to the cost data
- `visit_concept_id` to identify the standardized visit type
- `visit_start_date` for the visit date
- `visit_source_value` to connect the standardized visit back to the corresponding staging encounter

**`COST`**

This table provides the standardized financial information. I use:

- `cost_event_id` to connect the cost record to the visit
- `total_cost` for the total cost of the visit
- `total_paid` for payer coverage
- `paid_coinsurance` as estimated patient financial responsibility

In this ETL, `paid_coinsurance` is calculated as `total_cost - payer_coverage`.

**`stg.encounters`**

This table keeps the payer ID for each encounter. I use:

- `encounter_id` to connect the staging encounter to `VISIT_OCCURRENCE.visit_source_value`
- `payer_id` to connect the encounter to the payer table

**`stg.payers`**

This table provides the payer name. I use:

- `payer_id` to connect to the encounter
- `name` to identify the payer

The tables are connected as follows:

`VISIT_OCCURRENCE.visit_occurrence_id` → `COST.cost_event_id`

`VISIT_OCCURRENCE.visit_source_value` → `stg.encounters.encounter_id`

`stg.encounters.payer_id` → `stg.payers.payer_id`

These tables provide the visit type, payer, and financial fields needed to compare patient financial responsibility across different payer and visit-type combinations.

### 3.3 Build the Analysis Dataset

Now I have the pieces I need, so I put them into one table for the analysis.

I also join the OMOP concept table here. This changes the visit concept ID into a readable visit type, so I do not have to analyze numbers like 9201 or 9202.

In [70]:
# Build

omop_con.sql("""
    CREATE OR REPLACE TABLE analysis_dataset AS
    SELECT v.visit_occurrence_id, v.visit_start_date, v.visit_concept_id,
    concept.concept_name AS visit_type, p.name AS payer_name,
    c.total_cost, c.total_paid AS payer_coverage,
    c.paid_coinsurance AS estimated_patient_responsibility
    FROM omop__dev.visit_occurrence AS v

    INNER JOIN omop__dev.cost AS c
    ON v.visit_occurrence_id = c.cost_event_id
    AND c.cost_domain_id = 'Visit Occurrence'

    LEFT JOIN stg__dev.encounters AS e
    ON v.visit_source_value = e.encounter_id

    LEFT JOIN stg__dev.payers AS p
    ON e.payer_id = p.payer_id

    LEFT JOIN vocab__dev.concept AS concept
    ON v.visit_concept_id = concept.concept_id
""")

# Preview
omop_con.sql("""
SELECT *
FROM analysis_dataset
LIMIT 10
""").df()

,visit_occurrence_id,visit_start_date,visit_concept_id,visit_type,payer_name,total_cost,payer_coverage,estimated_patient_responsibility
0,8839,2021-04-15,9202,Outpatient Visit,UnitedHealthcare,277.27,0.00,277.27
1,8830,2019-06-13,9202,Outpatient Visit,UnitedHealthcare,690.86,552.69,138.17
2,8819,2018-08-02,9202,Outpatient Visit,Anthem,346.80,346.80,0.00
3,8811,2018-06-21,9202,Outpatient Visit,Anthem,722.67,722.67,0.00
4,8805,2018-05-16,9202,Outpatient Visit,Anthem,346.80,346.80,0.00
5,8794,2018-04-15,9202,Outpatient Visit,Anthem,346.80,346.80,0.00
6,8783,2018-03-15,9202,Outpatient Visit,Anthem,346.80,346.80,0.00
7,8775,2018-01-30,9202,Outpatient Visit,Anthem,593.27,474.62,118.65
8,8764,2017-12-30,9202,Outpatient Visit,Anthem,346.80,277.45,69.35
9,8756,2017-11-29,9202,Outpatient Visit,Anthem,743.74,594.99,148.75


In [71]:
# Final check
omop_con.sql("""
select count(*) total_rows,
 sum(visit_type is null) missing_visit_type,
 sum(payer_name is null) missing_payer,
 sum(total_cost is null) missing_cost,
 sum(payer_coverage is null) missing_coverage,
 sum(estimated_patient_responsibility is null) missing_patient_resp
from analysis_dataset
""").df()

,total_rows,missing_visit_type,missing_payer_name,missing_total_cost,missing_payer_coverage,missing_patient_responsibility
0,10652,0.0,0.0,0.0,0.0,0.0


The final table has 10,652 rows. The visit type, payer and financial fields I need do not have missing values.

At this point the data is ready for the business analysis.

### 3.4 Save the Analysis Table



In [73]:
# Export

omop_con.sql("""
COPY analysis_dataset TO '/content/analysis_dataset.csv'
 (HEADER, DELIMITER ',')
""")

print("saved to /content/analysis_dataset.csv")

Saved: /content/analysis_dataset.csv


## 4. Business Analysis

Now I can go back to my original question:

**Which payer and visit type combinations are associated with higher estimated patient financial responsibility?**

I started with a simple comparison of payer and visit type. I looked at visit count, average responsibility and median responsibility.

I included the median because the cost range was very wide when I checked the source data earlier.

In [74]:
payer_visit = omop_con.sql("""
select payer_name, visit_type,
 count(*) as encounter_count,
 round(avg(estimated_patient_responsibility),2) as avg_patient_resp,
 round(median(estimated_patient_responsibility),2) as median_patient_resp
from analysis_dataset
group by payer_name, visit_type
order by avg_patient_resp desc
""").df()

payer_visit.head(20)

,payer_name,visit_type,encounter_count,avg_patient_resp,median_patient_resp
0,NO_INSURANCE,Inpatient Visit,22,6445.28,2351.55
1,NO_INSURANCE,Outpatient Visit,507,5978.48,747.37
2,NO_INSURANCE,Emergency Room Visit,73,4213.15,1312.46
3,Humana,Inpatient Visit,21,2984.00,173.53
4,Cigna Health,Inpatient Visit,15,2761.51,143.40
5,Medicare,Inpatient Visit,13,2722.50,1060.33
6,Anthem,Inpatient Visit,14,2508.63,139.82
7,Blue Cross Blue Shield,Inpatient Visit,4,2165.23,1237.77
8,UnitedHealthcare,Inpatient Visit,8,1581.60,72.29
9,NO_INSURANCE,General examination of patient,202,1421.84,1224.23


#### First look

The first thing I noticed is that `NO_INSURANCE` is at the top, which makes sense because there is no payer coverage.

The more interesting part for me is the insured visits. Some inpatient groups have a high average, but their median is much lower.

For example, Humana inpatient has an average patient responsibility close to $3,000, but the median is only around 174.

That made me wonder if a small number of expensive visits are pulling the average up.

In [75]:
insured_dist = omop_con.sql("""
select payer_name, visit_type,
 count(*) encounter_count,
round(avg(estimated_patient_responsibility),2) avg_resp,
round(median(estimated_patient_responsibility),2) median_resp,
round(quantile_cont(estimated_patient_responsibility, .75),2) p75_resp,
round(max(estimated_patient_responsibility),2) max_resp
from analysis_dataset
where payer_name <> 'NO_INSURANCE'
group by payer_name, visit_type
order by avg_resp desc
""").df()

insured_dist.head(20)

,payer_name,visit_type,encounter_count,avg_resp,median_resp,p75_resp,max_resp
0,Humana,Inpatient Visit,21,2984.00,173.53,2895.11,19820.59
1,Cigna Health,Inpatient Visit,15,2761.51,143.40,2205.69,18430.07
2,Medicare,Inpatient Visit,13,2722.50,1060.33,5416.05,8600.89
3,Anthem,Inpatient Visit,14,2508.63,139.82,2457.54,12589.89
4,Blue Cross Blue Shield,Inpatient Visit,4,2165.23,1237.77,2672.17,6097.40
5,UnitedHealthcare,Inpatient Visit,8,1581.60,72.29,486.24,10795.34
6,Blue Cross Blue Shield,General examination of patient,59,1391.68,1224.23,1505.70,6620.36
7,Aetna,Inpatient Visit,15,1289.48,217.62,394.51,10308.84
8,Cigna Health,General examination of patient,153,1185.22,1188.20,1445.26,2531.42
9,Humana,General examination of patient,259,1161.41,1183.22,1474.35,2395.42


#### Finding

The insured payer groups showed two different patterns.

Some combinations, such as general examination visits for Cigna and Humana, had similar average and median patient responsibility, suggesting that higher responsibility was common across many visits.

Other combinations, especially inpatient and outpatient visits, had averages much higher than their medians and very high maximum values. This suggests that a smaller group of high-cost visits was driving much of the average.

In [76]:
resp_cutoff = omop_con.sql("""
select
 round(median(estimated_patient_responsibility),2) median_resp,
 round(quantile_cont(estimated_patient_responsibility,.75),2) p75_resp,
 round(quantile_cont(estimated_patient_responsibility,.90),2) p90_resp,
round(avg(estimated_patient_responsibility),2) avg_resp,
round(max(estimated_patient_responsibility),2) max_resp
from analysis_dataset
where payer_name != 'NO_INSURANCE'
""").df()

resp_cutoff

,median_resp,p75_resp,p90_resp,avg_resp,max_resp
0,84.96,446.1,1382.93,633.33,38370.84


#### High-Responsibility Threshold

Patient responsibility is highly skewed. Among insured visits, the median is $84.96, while the average is $633.33.

I use the 75 percentile ($446.10) as the threshold for a high-responsibility visit. This helps identify payer and visit-type combinations that have a higher share of financially risky visits.

In [77]:
high_resp = omop_con.sql("""
select payer_name, visit_type,
 count(*) as visits,
 sum(case when estimated_patient_responsibility >= 446.10 then 1 else 0 end) high_resp_visits,
 round(100.0 * sum(case when estimated_patient_responsibility >= 446.10 then 1 else 0 end) / count(*),1) as high_resp_rate
from analysis_dataset
where payer_name != 'NO_INSURANCE'
group by payer_name, visit_type
having count(*) >= 20
order by high_resp_rate desc
""").df()

high_resp.head(20)

,payer_name,visit_type,visits,high_resp_visits,high_resp_rate
0,Blue Cross Blue Shield,General examination of patient,59,53.0,89.8
1,Cigna Health,General examination of patient,153,136.0,88.9
2,Humana,General examination of patient,259,229.0,88.4
3,Aetna,General examination of patient,78,63.0,80.8
4,Anthem,General examination of patient,135,94.0,69.6
5,Medicare,General examination of patient,226,118.0,52.2
6,UnitedHealthcare,General examination of patient,54,27.0,50.0
7,Cigna Health,Emergency Room Visit,77,32.0,41.6
8,Humana,Inpatient Visit,21,8.0,38.1
9,Humana,Emergency Room Visit,176,66.0,37.5


#### Finding

General examination visits had the highest rate of high patient responsibility across almost every payer. For several commercial payers, more than 80% of these visits were above the $446.10 threshold.

Outpatient visits had lower high-responsibility rates, but some payer groups had much larger visit volumes. For example, Humana outpatient visits had a 30.8% high-responsibility rate, representing 427 high-responsibility visits.

In [78]:
visit_risk = omop_con.sql("""
select visit_type,
 count(*) visits,
 sum(case when estimated_patient_responsibility >= 446.10 then 1 else 0 end) as high_resp_visits,
 round(100.0 * sum(case when estimated_patient_responsibility >= 446.10 then 1 else 0 end) / count(*),1) high_resp_rate,
 round(median(estimated_patient_responsibility),2) median_resp
from analysis_dataset
where payer_name <> 'NO_INSURANCE'
group by visit_type
order by high_resp_rate desc
""").df()

visit_risk

,visit_type,visits,high_resp_visits,high_resp_rate,median_resp
0,General examination of patient,1088,720.0,66.2,921.18
1,Inpatient Visit,102,37.0,36.3,143.41
2,Emergency Room Visit,644,180.0,28.0,143.38
3,Outpatient Visit,8014,1532.0,19.1,65.06


#### Finding

Visit type showed a clear difference in high patient responsibility.

General examination visits had the highest high-responsibility rate at 66.2%, with a median patient responsibility of $921.18.
Outpatient visits had a much lower rate at 19.1%, but because of the much larger visit volume, they still accounted for 1,532 high-responsibility visits.

This suggests that both risk rate and visit volume should be considered when identifying financial-risk areas.

In [79]:
payer_risk = omop_con.sql("""
select payer_name,
 count(*) visits,
 sum(case when estimated_patient_responsibility >= 446.10 then 1 else 0 end) high_resp_visits,
 round(100.0 * sum(case when estimated_patient_responsibility >= 446.10 then 1 else 0 end)/count(*),1) as high_resp_rate,
 round(median(estimated_patient_responsibility),2) median_resp
   from analysis_dataset
where payer_name != 'NO_INSURANCE'
group by payer_name
order by high_resp_rate desc
""").df()

payer_risk

,payer_name,visits,high_resp_visits,high_resp_rate,median_resp
0,Humana,1843,730.0,39.6,143.41
1,Cigna Health,1098,430.0,39.2,206.30
2,Blue Cross Blue Shield,829,248.0,29.9,84.76
3,Anthem,992,280.0,28.2,142.35
4,Medicare,1253,324.0,25.9,242.97
5,Aetna,1368,319.0,23.3,0.00
6,UnitedHealthcare,1425,138.0,9.7,50.80
7,Dual Eligible,46,0.0,0.0,35.00
8,Medicaid,994,0.0,0.0,50.00


#### Finding

Patient responsibility also varied across payers: Humana and Cigna had the highest high-responsibility rates among insured visits, both around 39%. UnitedHealthcare was much lower at 9.7%, while Medicaid and Dual Eligible visits had no records above the $446.10 threshold in this dataset.

Together with the visit-type results, this suggests that both payer and visit type are useful for identifying higher-risk visits

In [80]:
priority = omop_con.sql("""
SELECT payer_name, visit_type,
       count(*) visits,
       sum(estimated_patient_responsibility >= 446.10) high_resp_visits,

       round(100 * sum(estimated_patient_responsibility >= 446.10) / count(*), 1
       ) high_resp_rate,

      round(sum(estimated_patient_responsibility), 2) total_patient_resp

FROM analysis_dataset

WHERE payer_name <> 'NO_INSURANCE'

GROUP BY payer_name, visit_type
HAVING count(*) >= 20

ORDER BY high_resp_visits DESC
""").df()

priority.head(15)

,payer_name,visit_type,visits,high_resp_visits,high_resp_rate,total_patient_resp
0,Humana,Outpatient Visit,1387,427.0,30.8,1554620.17
1,Cigna Health,Outpatient Visit,853,255.0,29.9,681974.42
2,Humana,General examination of patient,259,229.0,88.4,300804.99
3,Aetna,Outpatient Visit,1190,224.0,18.8,568560.17
4,Blue Cross Blue Shield,Outpatient Visit,747,184.0,24.6,408420.24
5,Medicare,Outpatient Visit,938,178.0,19.0,507252.65
6,Anthem,Outpatient Visit,756,163.0,21.6,522344.14
7,Cigna Health,General examination of patient,153,136.0,88.9,181339.15
8,Medicare,General examination of patient,226,118.0,52.2,238052.07
9,UnitedHealthcare,Outpatient Visit,1310,101.0,7.7,300796.85


#### Priority Finding

The payer and visit-type combinations showed two different priority patterns.

Humana outpatient visits had the largest number of high-responsibility visits, with 427 cases and more than $1.5 million in total estimated patient responsibility. This was mainly driven by the large visit volume.

General examination visits showed much higher high-responsibility rates across several payers. For example, Cigna, Humana, Aetna, and Blue Cross Blue Shield all had rates above 80%.

This suggests that high-volume outpatient groups and high-rate general examination groups may need different operational attention.

## 5. Business Insights and Recommendations

In my daily work, Epic already checks insurance eligibility before the patient's visit. If the patient is self-pay, we can also send a cost estimate before the appointment.

But having active insurance does not always mean the patient will have a low out-of-pocket cost. This is something I wanted to look at in this project.

The analysis shows that some insured patients still have a much higher chance of having high patient responsibility. General examination visits are a good example. For BCBS, Cigna, Humana, and Aetna, more than 80% of these visits were above the $446.10 threshold.

I think this information could be useful before the visit. Instead of only identifying self-pay patients, the system could also flag some insured visits with a higher financial risk. These patients could receive a cost estimate or cost information earlier, so they have more time to understand what they may need to pay.

I also found that high rate and high volume are different problems. Humana outpatient visits had a high-responsibility rate of 30.8%, but because there were many visits, this group still had 427 high-responsibility cases. I would not manually review every Humana outpatient visit. More information, such as deductible, plan type, or the service being provided, would be needed to narrow down which patients really need additional review.

From my work experience, I think this type of analysis is more useful as an extra step after the normal insurance verification, instead of replacing the current workflow. It can help the team decide which insured patients may need more financial information before their visit.